# Objective

Convert the extracted university document text into smaller overlapping chunks suitable for retrieval in a RAG system.

## Document Chunking
The objective of this notebook is to split extracted document text into smaller overlapping chunks suitable for Retrieval-Augmented Generation (RAG).

Large university documents cannot be passed directly to a Large Language Model because of context window limitations. Chunking divides the document into smaller sections while preserving context through overlap.

The generated chunks will later be converted into embeddings and stored in a FAISS vector database for retrieval.

In [1]:
# Install libraries required for document chunking

!pip install -q langchain

In [20]:
# Install LangChain and text splitters

!pip install -q langchain langchain-text-splitters

In [17]:
# Display available datasets to confirm the correct document path

import os

for item in os.listdir("/kaggle/input"):
    print(item)

datasets


In [18]:
# Import the PDF reader library

from pypdf import PdfReader


# Load the General Academic Regulations document
# PDF extraction was validated in Notebook 01.
# The document is reloaded here because chunking requires access to the full text.

pdf_path = "/kaggle/input/datasets/makhosazanemthethwa/cos802-university-documents/General_Academic_Regulations_2026.pdf"

reader = PdfReader(pdf_path)


# Extract text from all pages and combine it into a single document

full_text = ""

for page in reader.pages:

    page_text = page.extract_text()

    if page_text:
        full_text += page_text + "\n"


# Display document statistics before chunking

print(f"Characters: {len(full_text):,}")
print(f"Words: {len(full_text.split()):,}")

Characters: 361,882
Words: 53,391


In [21]:
# Import LangChain's recursive text splitter

from langchain_text_splitters import RecursiveCharacterTextSplitter


# Create overlapping chunks to preserve context between neighbouring sections

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)


# Split the document into retrieval-ready chunks

chunks = text_splitter.split_text(full_text)


# Display the total number of chunks generated

print(f"Chunks Created: {len(chunks)}")

Chunks Created: 455


In [22]:
# Calculate chunk length statistics to understand chunk distribution

chunk_lengths = [len(chunk) for chunk in chunks]

print(f"Minimum Chunk Length: {min(chunk_lengths)}")
print(f"Maximum Chunk Length: {max(chunk_lengths)}")
print(f"Average Chunk Length: {sum(chunk_lengths) / len(chunk_lengths):.2f}")

Minimum Chunk Length: 639
Maximum Chunk Length: 998
Average Chunk Length: 955.34


In [23]:
# Inspect the first chunk to verify chunk quality

print(chunks[0])

University of Pretoria 
Private Bag X20, Hatﬁeld 0028, South Africa 
Tel +27 (0)12 420 3111 
www.up.ac.za
Make today matter 
www.up.ac.za
Make Today Matter 
www.up.ac.za
General Academic Regulations 
and 
Student Rules 
2026
General Academic Regulations and Student Rules 2026 
 
GENERAL ACADEMIC REGULATIONS AND STUDENT RULES 
 
TABLE OF CONTENTS 
 
A message from the Director of Enrolment and Student Administration 
 
GENERAL ACADEMIC REGULATIONS ................................................................................. 1 
SECTION A: INTRODUCTION AND GENERAL PROVISIONS .............................................. 1 
A1 Status and interpretation of the General Academic Regulations .................................... 1 
A2 Faculty regulations ........................................................................................................ 2 
A3 Application, selection and admission ............................................................................. 3


In [24]:
# Inspect a chunk from the middle of the document

middle_chunk = len(chunks) // 2

print(chunks[middle_chunk])

General Academic Regulations 2026 
51 
(e) If the dean considers it advisable for the faculty, the candidate may be required 
to present a seminar to the department in support of the application. In this case, 
the head of department should include a report on this in their recommendation. 
 
3.2  The application of the candidate, together with the reports and recommendations, is 
submitted for consideration to the dean (who may delegate the responsibility to the 
research and/or postgraduate committee). The recommendation of the research 
and/or postgraduate committee should be submitted to the relevant faculty board for 
recommendation for approval to the Senate Committee for Research and 
Postgraduate Education. The approval of the conversion must be submitted to Senate 
for noting. 
 
 
SECTION E: DOCTORAL DEGREES (G42–G55) 
 
GE1 Types: The Higher Education Qualifications Sub- Framework (HEQSF) distinguishes two


In [26]:
# Inspect a random chunk from the document

import random

random_chunk = random.choice(chunks)

print(random_chunk)

SECTION C:  HONOURS DEGREES AND POSTGRADUATE DIPLOMAS (G16–G29) 
 
GC1 Scope: These General Academic Regulations, with the necessary changes, also apply to 
postgraduate diplomas. 
 
GC2 Level: The honours degree and the postgraduate diploma are postgraduate specialisation 
qualifications at NQF-level 8 of at least 120 credits that prepare students for research-based 
postgraduate study. 
 
GC3 Articulation: An honours degree and a postgraduate diploma both constitute the minimum 
entry requirement for admission to an appropriate master’s degree. 
 
G16 Admission 
 
1. Compliance with the minimum requirements set out below will not guarantee admission to an 
honours degree at the University. In addition to institutional requirements, programme-
specific admission requirements are determined by the relevant faculty board, approved by 
Senate and contained in the relevant faculty yearbook. Furthermore, specific admission


## Observations
The General Academic Regulations and Student Rules 2026 document was successfully divided into 455 overlapping text chunks using the Recursive Character Text Splitter.

The generated chunks were generally consistent in size, with a minimum length of 639 characters, a maximum length of 998 characters and an average length of 955.34 characters. These values indicate that the chunking strategy adhered closely to the target chunk size of 1000 characters.

Inspection of chunks from the beginning, middle and random sections of the document showed that document structure, headings and semantic context were preserved. Examples included sections relating to doctoral degrees, honours degrees and postgraduate diplomas, demonstrating that the chunking process retained meaningful academic and regulatory information.

The results suggest that the selected chunk size and overlap configuration produced retrieval-ready chunks that are suitable for embedding generation and semantic search. The chunk overlap is expected to improve retrieval performance by preserving contextual information between neighbouring chunks.

Overall, the chunking process successfully transformed a large university regulations document containing approximately 53,391 words into a collection of manageable text units suitable for the next stage of the RAG pipeline.

## Conclusion
The General Academic Regulations document was successfully converted into 455 overlapping text chunks.

Inspection of chunks from the beginning, middle, and random positions within the document showed that document structure, headings, and semantic meaning were preserved.

The resulting chunks are suitable for embedding generation and retrieval in the next stage of the Retrieval-Augmented Generation (RAG) pipeline.